# BikeNodePlanner–OSM overlap

In [ ]:
# Run once if your environment is missing packages:
#%pip install geopandas osmnx matplotlib pandas shapely pyogrio
#pixi add pyrosm pyarrow

## Setup

In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import shapely
from IPython.display import display
from shapely.geometry import box
import requests
from pyrosm import OSM

working_dir = Path.cwd().resolve()

PROJECT_ROOT = working_dir.parent if working_dir.name == "osm_data_exploration" else working_dir

DENMARK_OSM_DIR = PROJECT_ROOT / "osm_data_exploration/outputs/denmark"
PBF_PATH = DENMARK_OSM_DIR / "denmark-latest.osm.pbf"

DENMARK_OSM_DIR.mkdir(parents=True, exist_ok=True)

EXTRACTION_MARGIN_M = 5_000
TOLERANCE_M = 20

## Load OSM Bike network

Tag information for Denmark: [LINK](https://taginfo.geofabrik.de/europe:denmark/)

In [ ]:
PBF_URL = ("https://download.geofabrik.de/europe/denmark-latest.osm.pbf")

if not PBF_PATH.exists():
    print("Downloading Denmark OSM data...")
    print("This can take several minutes.")

    temporary_path = Path(str(PBF_PATH) + ".download")

    with requests.get(PBF_URL, stream=True, timeout=120) as response:
        response.raise_for_status()
        total_bytes = int(response.headers.get("content-length", 0))
        downloaded_bytes = 0

        with temporary_path.open("wb") as file:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if not chunk:
                    continue

                file.write(chunk)
                downloaded_bytes += len(chunk)

                if total_bytes:
                    percentage = (100 * downloaded_bytes / total_bytes)
                    print(f"\rDownloaded: {percentage:5.1f}%", end="")

    temporary_path.replace(PBF_PATH)
    print("\nDownload completed.")

else:
    print("Using existing Denmark OSM file.")

print("OSM file:", PBF_PATH)

## Load plotting functions

Shared plotting functions are imported from `network_plotting.py`.


In [ ]:
import sys

plotting_module_dir = PROJECT_ROOT / "osm_data_exploration"
if str(plotting_module_dir) not in sys.path:
    sys.path.insert(0, str(plotting_module_dir))

from network_plotting import (
    plot_bicycle_networks,
    plot_bicycle_networks_with_export,
    plot_overlap_results,
    plot_matched_attribute,
)


In [ ]:
def group_lit(value):
    """
    Group OSM lit values into:
    missing, no or yes.
    """

    if pd.isna(value):
        return "missing"

    value = str(value).strip().lower()

    if value in {"yes", "automatic"}:
        return "yes"

    if value == "no":
        return "no"

    return "missing"


def group_surface(value):
    """
    Group OSM surface values into:
    missing, unpaved or paved.
    """

    if pd.isna(value):
        return "missing"

    value = str(value).strip().lower()

    paved_values = {
        "asphalt",
        "paved",
        "concrete",
        "concrete:lanes",
        "concrete:plates",
        "paving_stones",
        "sett",
        "cobblestone",
        "metal",
        "wood",
        "grass_paver",
    }

    unpaved_values = {
        "unpaved",
        "gravel",
        "fine_gravel",
        "compacted",
        "pebblestone",
        "ground",
        "dirt",
        "earth",
        "grass",
        "sand",
        "mud",
        "rock",
        "scree",
    }

    if value in paved_values:
        return "paved"

    if value in unpaved_values:
        return "unpaved"

    return "missing"

# Analyse each area separately

## Bornholm

In [ ]:
SAMPLE = "bornholm"

BIKE_LOOPS_DIR = PROJECT_ROOT / f"bikenwloops/data/input/{SAMPLE}/network/processed"
OUTPUT_DIR = PROJECT_ROOT / f"osm_data_exploration/outputs/{SAMPLE}"

OSM_EDGES_PATH = PROJECT_ROOT / f"osm_data_exploration/outputs/{SAMPLE}/{SAMPLE}_osm_bike_edges.parquet"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OSM_EDGES_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Bike loop network directory:", BIKE_LOOPS_DIR)
print("Output folder:", OUTPUT_DIR)

In [ ]:
# Load bike loop network data
bike_loops_edges = gpd.read_file(BIKE_LOOPS_DIR / "edges_slope.gpkg")
bike_loops_nodes = gpd.read_file(BIKE_LOOPS_DIR / "nodes.gpkg")

bike_loops_edges = bike_loops_edges.loc[
    bike_loops_edges.geometry.notna()
    & ~bike_loops_edges.geometry.is_empty
].copy()

bike_loops_nodes = bike_loops_nodes.loc[
    bike_loops_nodes.geometry.notna()
    & ~bike_loops_nodes.geometry.is_empty
].copy()

if bike_loops_edges.crs is None:
    raise ValueError("The bike-loop edges do not have a CRS.")

BIKE_LOOPS_CRS = bike_loops_edges.crs
bike_loops_nodes = bike_loops_nodes.to_crs(BIKE_LOOPS_CRS)

print("Bike loops CRS:", BIKE_LOOPS_CRS)
print(f"Bike loop edges: {len(bike_loops_edges):,}")
print(f"Bike loop nodes: {len(bike_loops_nodes):,}")

In [ ]:
minx, miny, maxx, maxy = bike_loops_edges.total_bounds

bike_loops_bbox = box(
    minx - EXTRACTION_MARGIN_M,
    miny - EXTRACTION_MARGIN_M,
    maxx + EXTRACTION_MARGIN_M,
    maxy + EXTRACTION_MARGIN_M,
)

# The bounding box below uses the BikeNodePlanner coordinate system (Bike loops CRS: EPSG:25832) with coordinates measured in metres. 
# This is useful for applying the 5,000-metre margin and performing the 20-metre overlap analysis.
bike_loops_bbox_gdf = gpd.GeoDataFrame(
    geometry=[bike_loops_bbox],
    crs=BIKE_LOOPS_CRS,
)

# WGS84 means the bounding box coordinates have been converted to the standard global longitude/latitude coordinate system, EPSG:4326.
bike_loops_bbox_wgs84 = (
    bike_loops_bbox_gdf
    .to_crs("EPSG:4326")
    .geometry
    .iloc[0]
)

### Extract area of interest from OSM data

In [ ]:
required_osm_columns = {
    "geometry",
    "lit",
    "surface",
}

rebuild_osm_cache = True

### Time: ≈ 2 min
if OSM_EDGES_PATH.exists():
    osm_edges = gpd.read_parquet(OSM_EDGES_PATH)

    missing_columns = (required_osm_columns - set(osm_edges.columns))

    if not missing_columns:
        rebuild_osm_cache = False
        print(f"Loaded cached {SAMPLE} OSM bicycle network.")

    else:
        print(f"The cached file is missing: {sorted(missing_columns)}")
        print("Rebuilding the OSM cache...")

if rebuild_osm_cache:
    print(f"Extracting the {SAMPLE} bicycle network from the Denmark PBF...")

    osm_reader = OSM(str(PBF_PATH), bounding_box=bike_loops_bbox_wgs84)

    osm_edges = osm_reader.get_network(
        network_type="cycling",
        extra_attributes=["lit", "surface"],
    )

    osm_edges = osm_edges.loc[
        osm_edges.geometry.notna()
        & ~osm_edges.geometry.is_empty
    ].copy()

    osm_edges = osm_edges.explode(
        index_parts=False,
        ignore_index=True,
    )

    osm_edges = osm_edges.to_crs(BIKE_LOOPS_CRS)

    columns_to_keep = [
        column
        for column in [
            "id",
            "name",
            "highway",
            "lit",
            "surface",
            "geometry",
        ]
        if column in osm_edges.columns
    ]

    osm_edges = osm_edges[columns_to_keep].copy()
    osm_edges.to_parquet(OSM_EDGES_PATH)
    print(f"Saved {SAMPLE.title()} OSM bicycle network.")

print(f"OSM bicycle edges: {len(osm_edges):,}")
print("OSM CRS:", osm_edges.crs)

In [ ]:
print("Lit values:")

display(
    osm_edges["lit"]
    .fillna("missing")
    .value_counts()
    .to_frame("edges")
)

print("Surface values:")

display(
    osm_edges["surface"]
    .fillna("missing")
    .value_counts()
    .to_frame("edges")
)

In [ ]:
plot_bicycle_networks(
    sample=SAMPLE,
    bike_loops_edges=bike_loops_edges,
    bike_loops_nodes=bike_loops_nodes,
    bike_loops_bbox_gdf=bike_loops_bbox_gdf,
    osm_edges=osm_edges,
)

### Overlap analysis of the two bike networks

In [ ]:
osm_spatial_index = osm_edges.sindex
matched_geometries, unmatched_geometries, matched_lengths, osm_candidate_counts = [], [], [], []

matched_osm_ids, matched_lit_values, matched_surface_values = [], [], []
number_of_edges = len(bike_loops_edges)

for position, geometry in enumerate(bike_loops_edges.geometry, start=1):

    # Find only OSM lines within the matching tolerance
    candidate_positions = osm_spatial_index.query(
        geometry,
        predicate="dwithin", # This predicate finds geometries that are within a specified distance of the input geometry.
        distance=TOLERANCE_M,
    )

    candidate_positions = np.unique(candidate_positions)
    
    candidates = osm_edges.iloc[candidate_positions]

    if candidates.empty:
        matched_geometry = shapely.GeometryCollection()
        unmatched_geometry = geometry

        matched_osm_id = np.nan
        lit_value = np.nan
        surface_value = np.nan

    else:
        # Calculate the complete matched geometry
        local_osm_network = shapely.union_all(candidates.geometry.array)
        local_osm_buffer = local_osm_network.buffer(TOLERANCE_M)
        matched_geometry = geometry.intersection(local_osm_buffer)
        unmatched_geometry = geometry.difference(local_osm_buffer)
        
        # Find which candidate covers the greatest
        # length of this bike-loop edge
        candidate_buffers = (
            candidates.geometry.buffer(
                TOLERANCE_M
            )
        )

        candidate_overlap_lengths = (
            candidate_buffers
            .intersection(geometry)
            .length
        )

        best_position = (
            candidate_overlap_lengths
            .to_numpy()
            .argmax()
        )

        best_candidate = candidates.iloc[best_position]

        matched_osm_id = best_candidate.get("id", np.nan)
        lit_value = best_candidate.get("lit", np.nan)
        surface_value = best_candidate.get("surface", np.nan)

    matched_geometries.append(matched_geometry)
    unmatched_geometries.append(unmatched_geometry)
    matched_lengths.append(matched_geometry.length)
    osm_candidate_counts.append(len(candidates))
    matched_osm_ids.append(matched_osm_id)
    matched_lit_values.append(lit_value)
    matched_surface_values.append(surface_value)

    if position % 500 == 0:
        print(f"Processed {position:,} of {number_of_edges:,} bike-loop edges")

print("Matching completed.")

In [ ]:
bike_loops_overlap = bike_loops_edges.copy()
bike_loops_overlap["bike_loop_length_m"] = (bike_loops_overlap.geometry.length)
bike_loops_overlap["matched_length_m"] = (matched_lengths)
bike_loops_overlap["unmatched_length_m"] = (bike_loops_overlap["bike_loop_length_m"] - bike_loops_overlap["matched_length_m"])
bike_loops_overlap["coverage_pct"] = (100 * bike_loops_overlap["matched_length_m"] / bike_loops_overlap["bike_loop_length_m"])
bike_loops_overlap["osm_candidate_count"] = (osm_candidate_counts)
bike_loops_overlap["matched_osm_id"] = (matched_osm_ids)
bike_loops_overlap["lit"] = (matched_lit_values)
bike_loops_overlap["surface"] = (matched_surface_values)

bike_loops_overlap["lit_group"] = (
    bike_loops_overlap["lit"].apply(
        group_lit
    )
)

bike_loops_overlap["surface_group"] = (
    bike_loops_overlap["surface"].apply(
        group_surface
    )
)

result_columns = [
    column
    for column in [
        "edge_id",
        "node_start",
        "node_end",
        "bike_loop_length_m",
        "matched_length_m",
        "unmatched_length_m",
        "coverage_pct",
        "osm_candidate_count",
        "matched_osm_id",
        "lit",
        "lit_group",
        "surface",
        "surface_group",
    ]
    if column in bike_loops_overlap.columns
]

matched_edges = gpd.GeoDataFrame(
    bike_loops_overlap[result_columns].copy(),
    geometry=matched_geometries,
    crs=BIKE_LOOPS_CRS,
)

unmatched_edges = gpd.GeoDataFrame(
    bike_loops_overlap[result_columns].copy(),
    geometry=unmatched_geometries,
    crs=BIKE_LOOPS_CRS,
)

matched_edges = matched_edges.loc[~matched_edges.geometry.is_empty].copy()
unmatched_edges = unmatched_edges.loc[~unmatched_edges.geometry.is_empty].copy()

print(f"Matched edges: {len(matched_edges):,}")
print(f"Edges with unmatched geometry: {len(unmatched_edges):,}")

In [ ]:
overlap_summary = plot_overlap_results(
    sample=SAMPLE,
    tolerance_m=TOLERANCE_M,
    output_dir=OUTPUT_DIR,
    bike_loops_edges=bike_loops_edges,
    bike_loops_nodes=bike_loops_nodes,
    osm_edges=osm_edges,
    bike_loops_overlap=bike_loops_overlap,
    matched_edges=matched_edges,
    unmatched_edges=unmatched_edges,
    plot_padding_m=1_000,
)

In [ ]:
fig, axes = plot_matched_attribute(
    sample=SAMPLE,
    osm_edges=osm_edges,
    matched_edges=matched_edges,
    unmatched_edges=unmatched_edges,
    bike_loops_edges=bike_loops_edges,
    bike_loops_nodes=bike_loops_nodes,
    tolerance_m=TOLERANCE_M,
    output_dir=OUTPUT_DIR,
)

## Fyn

In [ ]:
SAMPLE = "funen"

BIKE_LOOPS_DIR = PROJECT_ROOT / f"bikenwloops/data/input/{SAMPLE}/network/processed"
OUTPUT_DIR = PROJECT_ROOT / f"osm_data_exploration/outputs/{SAMPLE}"

OSM_EDGES_PATH = PROJECT_ROOT / f"osm_data_exploration/outputs/{SAMPLE}/{SAMPLE}_osm_bike_edges.parquet"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OSM_EDGES_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Bike loop network directory:", BIKE_LOOPS_DIR)
print("Output folder:", OUTPUT_DIR)

In [ ]:
# Load bike loop network data
bike_loops_edges = gpd.read_file(BIKE_LOOPS_DIR / "edges_slope.gpkg")
bike_loops_nodes = gpd.read_file(BIKE_LOOPS_DIR / "nodes.gpkg")

bike_loops_edges = bike_loops_edges.loc[
    bike_loops_edges.geometry.notna()
    & ~bike_loops_edges.geometry.is_empty
].copy()

bike_loops_nodes = bike_loops_nodes.loc[
    bike_loops_nodes.geometry.notna()
    & ~bike_loops_nodes.geometry.is_empty
].copy()

if bike_loops_edges.crs is None:
    raise ValueError("The bike-loop edges do not have a CRS.")

BIKE_LOOPS_CRS = bike_loops_edges.crs
bike_loops_nodes = bike_loops_nodes.to_crs(BIKE_LOOPS_CRS)

print("Bike loops CRS:", BIKE_LOOPS_CRS)
print(f"Bike loop edges: {len(bike_loops_edges):,}")
print(f"Bike loop nodes: {len(bike_loops_nodes):,}")

In [ ]:
minx, miny, maxx, maxy = bike_loops_edges.total_bounds

bike_loops_bbox = box(
    minx - EXTRACTION_MARGIN_M,
    miny - EXTRACTION_MARGIN_M,
    maxx + EXTRACTION_MARGIN_M,
    maxy + EXTRACTION_MARGIN_M,
)

# The bounding box below uses the BikeNodePlanner coordinate system (Bike loops CRS: EPSG:25832) with coordinates measured in metres. 
# This is useful for applying the 5,000-metre margin and performing the 20-metre overlap analysis.
bike_loops_bbox_gdf = gpd.GeoDataFrame(
    geometry=[bike_loops_bbox],
    crs=BIKE_LOOPS_CRS,
)

# WGS84 means the bounding box coordinates have been converted to the standard global longitude/latitude coordinate system, EPSG:4326.
bike_loops_bbox_wgs84 = (
    bike_loops_bbox_gdf
    .to_crs("EPSG:4326")
    .geometry
    .iloc[0]
)

### Extract area of interest from OSM data

In [ ]:
required_osm_columns = {
    "geometry",
    "lit",
    "surface",
}

rebuild_osm_cache = True

### Time: ≈ 8 min
if OSM_EDGES_PATH.exists():
    osm_edges = gpd.read_parquet(OSM_EDGES_PATH)

    missing_columns = (required_osm_columns - set(osm_edges.columns))

    if not missing_columns:
        rebuild_osm_cache = False
        print(f"Loaded cached {SAMPLE} OSM bicycle network.")

    else:
        print(f"The cached file is missing: {sorted(missing_columns)}")
        print("Rebuilding the OSM cache...")

if rebuild_osm_cache:
    print(f"Extracting the {SAMPLE} bicycle network from the Denmark PBF...")

    osm_reader = OSM(str(PBF_PATH), bounding_box=bike_loops_bbox_wgs84)

    osm_edges = osm_reader.get_network(
        network_type="cycling",
        extra_attributes=["lit", "surface"],
    )

    osm_edges = osm_edges.loc[
        osm_edges.geometry.notna()
        & ~osm_edges.geometry.is_empty
    ].copy()

    osm_edges = osm_edges.explode(
        index_parts=False,
        ignore_index=True,
    )

    osm_edges = osm_edges.to_crs(BIKE_LOOPS_CRS)

    columns_to_keep = [
        column
        for column in [
            "id",
            "name",
            "highway",
            "lit",
            "surface",
            "geometry",
        ]
        if column in osm_edges.columns
    ]

    osm_edges = osm_edges[columns_to_keep].copy()
    osm_edges.to_parquet(OSM_EDGES_PATH)
    print(f"Saved {SAMPLE.title()} OSM bicycle network.")

print(f"OSM bicycle edges: {len(osm_edges):,}")
print("OSM CRS:", osm_edges.crs)

In [ ]:
plot_bicycle_networks(
    sample=SAMPLE,
    bike_loops_edges=bike_loops_edges,
    bike_loops_nodes=bike_loops_nodes,
    bike_loops_bbox_gdf=bike_loops_bbox_gdf,
    osm_edges=osm_edges,
)

### Overlap analysis of the two bike networks

In [ ]:
osm_spatial_index = osm_edges.sindex
matched_geometries, unmatched_geometries, matched_lengths, osm_candidate_counts = [], [], [], []

matched_osm_ids, matched_lit_values, matched_surface_values = [], [], []

number_of_edges = len(bike_loops_edges)

for position, geometry in enumerate(bike_loops_edges.geometry, start=1):

    # Find only OSM lines within the matching tolerance
    candidate_positions = osm_spatial_index.query(
        geometry,
        predicate="dwithin", # This predicate finds geometries that are within a specified distance of the input geometry.
        distance=TOLERANCE_M,
    )

    candidate_positions = np.unique(candidate_positions)
    candidates = osm_edges.iloc[candidate_positions]

    if candidates.empty:
        matched_geometry = shapely.GeometryCollection()
        unmatched_geometry = geometry

        matched_osm_id = np.nan
        lit_value = np.nan
        surface_value = np.nan

    else:
        # Calculate the complete matched geometry
        local_osm_network = shapely.union_all(candidates.geometry.array)
        local_osm_buffer = local_osm_network.buffer(TOLERANCE_M)
        matched_geometry = geometry.intersection(local_osm_buffer)
        unmatched_geometry = geometry.difference(local_osm_buffer)
        
        # Find which candidate covers the greatest
        # length of this bike-loop edge
        candidate_buffers = (
            candidates.geometry.buffer(
                TOLERANCE_M
            )
        )

        candidate_overlap_lengths = (
            candidate_buffers
            .intersection(geometry)
            .length
        )

        best_position = (
            candidate_overlap_lengths
            .to_numpy()
            .argmax()
        )

        best_candidate = candidates.iloc[best_position]

        matched_osm_id = best_candidate.get("id", np.nan)
        lit_value = best_candidate.get("lit", np.nan)
        surface_value = best_candidate.get("surface", np.nan)

    matched_geometries.append(matched_geometry)
    unmatched_geometries.append(unmatched_geometry)
    matched_lengths.append(matched_geometry.length)
    osm_candidate_counts.append(len(candidates))
    matched_osm_ids.append(matched_osm_id)
    matched_lit_values.append(lit_value)
    matched_surface_values.append(surface_value)

    if position % 500 == 0:
        print(f"Processed {position:,} of {number_of_edges:,} bike-loop edges")

print("Matching completed.")

In [ ]:
bike_loops_overlap = bike_loops_edges.copy()
bike_loops_overlap["bike_loop_length_m"] = (bike_loops_overlap.geometry.length)
bike_loops_overlap["matched_length_m"] = (matched_lengths)
bike_loops_overlap["unmatched_length_m"] = (bike_loops_overlap["bike_loop_length_m"] - bike_loops_overlap["matched_length_m"])
bike_loops_overlap["coverage_pct"] = (100 * bike_loops_overlap["matched_length_m"] / bike_loops_overlap["bike_loop_length_m"])
bike_loops_overlap["osm_candidate_count"] = (osm_candidate_counts)
bike_loops_overlap["matched_osm_id"] = (matched_osm_ids)
bike_loops_overlap["lit"] = (matched_lit_values)
bike_loops_overlap["surface"] = (matched_surface_values)

bike_loops_overlap["lit_group"] = (
    bike_loops_overlap["lit"].apply(
        group_lit
    )
)

bike_loops_overlap["surface_group"] = (
    bike_loops_overlap["surface"].apply(
        group_surface
    )
)

result_columns = [
    column
    for column in [
        "edge_id",
        "node_start",
        "node_end",
        "bike_loop_length_m",
        "matched_length_m",
        "unmatched_length_m",
        "coverage_pct",
        "osm_candidate_count",
        "matched_osm_id",
        "lit",
        "lit_group",
        "surface",
        "surface_group",
    ]
    if column in bike_loops_overlap.columns
]

matched_edges = gpd.GeoDataFrame(
    bike_loops_overlap[result_columns].copy(),
    geometry=matched_geometries,
    crs=BIKE_LOOPS_CRS,
)

unmatched_edges = gpd.GeoDataFrame(
    bike_loops_overlap[result_columns].copy(),
    geometry=unmatched_geometries,
    crs=BIKE_LOOPS_CRS,
)

matched_edges = matched_edges.loc[~matched_edges.geometry.is_empty].copy()
unmatched_edges = unmatched_edges.loc[~unmatched_edges.geometry.is_empty].copy()

print(f"Matched edges: {len(matched_edges):,}")
print(f"Edges with unmatched geometry: {len(unmatched_edges):,}")

In [ ]:
overlap_summary = plot_overlap_results(
    sample=SAMPLE,
    tolerance_m=TOLERANCE_M,
    output_dir=OUTPUT_DIR,
    bike_loops_edges=bike_loops_edges,
    bike_loops_nodes=bike_loops_nodes,
    osm_edges=osm_edges,
    bike_loops_overlap=bike_loops_overlap,
    matched_edges=matched_edges,
    unmatched_edges=unmatched_edges,
    plot_padding_m=1_000,
)

In [ ]:
fig, axes = plot_matched_attribute(
    sample=SAMPLE,
    osm_edges=osm_edges,
    matched_edges=matched_edges,
    unmatched_edges=unmatched_edges,
    bike_loops_edges=bike_loops_edges,
    bike_loops_nodes=bike_loops_nodes,
    tolerance_m=TOLERANCE_M,
    output_dir=OUTPUT_DIR,
)

## Jylland

In [ ]:
SAMPLE = "jutland"

BIKE_LOOPS_DIR = PROJECT_ROOT / f"bikenwloops/data/input/{SAMPLE}/network/processed"
OUTPUT_DIR = PROJECT_ROOT / f"osm_data_exploration/outputs/{SAMPLE}"

OSM_EDGES_PATH = PROJECT_ROOT / f"osm_data_exploration/outputs/{SAMPLE}/{SAMPLE}_osm_bike_edges.parquet"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OSM_EDGES_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Bike loop network directory:", BIKE_LOOPS_DIR)
print("Output folder:", OUTPUT_DIR)

In [ ]:
# Load bike loop network data
bike_loops_edges = gpd.read_file(BIKE_LOOPS_DIR / "edges_slope.gpkg")
bike_loops_nodes = gpd.read_file(BIKE_LOOPS_DIR / "nodes.gpkg")

bike_loops_edges = bike_loops_edges.loc[
    bike_loops_edges.geometry.notna()
    & ~bike_loops_edges.geometry.is_empty
].copy()

bike_loops_nodes = bike_loops_nodes.loc[
    bike_loops_nodes.geometry.notna()
    & ~bike_loops_nodes.geometry.is_empty
].copy()

if bike_loops_edges.crs is None:
    raise ValueError("The bike-loop edges do not have a CRS.")

BIKE_LOOPS_CRS = bike_loops_edges.crs
bike_loops_nodes = bike_loops_nodes.to_crs(BIKE_LOOPS_CRS)

print("Bike loops CRS:", BIKE_LOOPS_CRS)
print(f"Bike loop edges: {len(bike_loops_edges):,}")
print(f"Bike loop nodes: {len(bike_loops_nodes):,}")

In [ ]:
minx, miny, maxx, maxy = bike_loops_edges.total_bounds

bike_loops_bbox = box(
    minx - EXTRACTION_MARGIN_M,
    miny - EXTRACTION_MARGIN_M,
    maxx + EXTRACTION_MARGIN_M,
    maxy + EXTRACTION_MARGIN_M,
)

# The bounding box below uses the BikeNodePlanner coordinate system (Bike loops CRS: EPSG:25832) with coordinates measured in metres. 
# This is useful for applying the 5,000-metre margin and performing the 20-metre overlap analysis.
bike_loops_bbox_gdf = gpd.GeoDataFrame(
    geometry=[bike_loops_bbox],
    crs=BIKE_LOOPS_CRS,
)

# WGS84 means the bounding box coordinates have been converted to the standard global longitude/latitude coordinate system, EPSG:4326.
bike_loops_bbox_wgs84 = (
    bike_loops_bbox_gdf
    .to_crs("EPSG:4326")
    .geometry
    .iloc[0]
)

### Extract area of interest from OSM data

In [ ]:
required_osm_columns = {
    "geometry",
    "lit",
    "surface",
}

rebuild_osm_cache = True

### Time: ≈ 50 min
if OSM_EDGES_PATH.exists():
    osm_edges = gpd.read_parquet(OSM_EDGES_PATH)
    missing_columns = (required_osm_columns - set(osm_edges.columns))
    
    if not missing_columns:
        rebuild_osm_cache = False
        print(f"Loaded cached {SAMPLE} OSM bicycle network.")

    else:
        print(f"The cached file is missing: {sorted(missing_columns)}")
        print("Rebuilding the OSM cache...")

if rebuild_osm_cache:
    print(f"Extracting the {SAMPLE} bicycle network from the Denmark PBF...")

    osm_reader = OSM(str(PBF_PATH), bounding_box=bike_loops_bbox_wgs84)

    osm_edges = osm_reader.get_network(
        network_type="cycling",
        extra_attributes=["lit", "surface"],
    )

    osm_edges = osm_edges.loc[
        osm_edges.geometry.notna()
        & ~osm_edges.geometry.is_empty
    ].copy()

    osm_edges = osm_edges.explode(
        index_parts=False,
        ignore_index=True,
    )

    osm_edges = osm_edges.to_crs(BIKE_LOOPS_CRS)

    columns_to_keep = [
        column
        for column in [
            "id",
            "name",
            "highway",
            "lit",
            "surface",
            "geometry",
        ]
        if column in osm_edges.columns
    ]

    osm_edges = osm_edges[columns_to_keep].copy()
    osm_edges.to_parquet(OSM_EDGES_PATH)
    print(f"Saved {SAMPLE.title()} OSM bicycle network.")

print(f"OSM bicycle edges: {len(osm_edges):,}")
print("OSM CRS:", osm_edges.crs)

In [ ]:
plot_bicycle_networks(
    sample=SAMPLE,
    bike_loops_edges=bike_loops_edges,
    bike_loops_nodes=bike_loops_nodes,
    bike_loops_bbox_gdf=bike_loops_bbox_gdf,
    osm_edges=osm_edges,
)

### Overlap analysis of the two bike networks

In [ ]:
osm_spatial_index = osm_edges.sindex
matched_geometries, unmatched_geometries, matched_lengths, osm_candidate_counts = [], [], [], []

matched_osm_ids, matched_lit_values, matched_surface_values = [], [], []
number_of_edges = len(bike_loops_edges)

for position, geometry in enumerate(bike_loops_edges.geometry, start=1):

    # Find only OSM lines within the matching tolerance
    candidate_positions = osm_spatial_index.query(
        geometry,
        predicate="dwithin", # This predicate finds geometries that are within a specified distance of the input geometry.
        distance=TOLERANCE_M,
    )

    candidate_positions = np.unique(candidate_positions)
    
    candidates = osm_edges.iloc[candidate_positions]

    if candidates.empty:
        matched_geometry = shapely.GeometryCollection()
        unmatched_geometry = geometry

        matched_osm_id = np.nan
        lit_value = np.nan
        surface_value = np.nan

    else:
        # Calculate the complete matched geometry
        local_osm_network = shapely.union_all(candidates.geometry.array)
        local_osm_buffer = local_osm_network.buffer(TOLERANCE_M)
        matched_geometry = geometry.intersection(local_osm_buffer)
        unmatched_geometry = geometry.difference(local_osm_buffer)
        
        # Find which candidate covers the greatest
        # length of this bike-loop edge
        candidate_buffers = (
            candidates.geometry.buffer(
                TOLERANCE_M
            )
        )

        candidate_overlap_lengths = (
            candidate_buffers
            .intersection(geometry)
            .length
        )

        best_position = (
            candidate_overlap_lengths
            .to_numpy()
            .argmax()
        )

        best_candidate = candidates.iloc[best_position]

        matched_osm_id = best_candidate.get("id", np.nan)
        lit_value = best_candidate.get("lit", np.nan)
        surface_value = best_candidate.get("surface", np.nan)

    matched_geometries.append(matched_geometry)
    unmatched_geometries.append(unmatched_geometry)
    matched_lengths.append(matched_geometry.length)
    osm_candidate_counts.append(len(candidates))
    matched_osm_ids.append(matched_osm_id)
    matched_lit_values.append(lit_value)
    matched_surface_values.append(surface_value)

    if position % 500 == 0:
        print(f"Processed {position:,} of {number_of_edges:,} bike-loop edges")


In [ ]:
bike_loops_overlap = bike_loops_edges.copy()
bike_loops_overlap["bike_loop_length_m"] = (bike_loops_overlap.geometry.length)
bike_loops_overlap["matched_length_m"] = (matched_lengths)
bike_loops_overlap["unmatched_length_m"] = (bike_loops_overlap["bike_loop_length_m"] - bike_loops_overlap["matched_length_m"])
bike_loops_overlap["coverage_pct"] = (100 * bike_loops_overlap["matched_length_m"] / bike_loops_overlap["bike_loop_length_m"])
bike_loops_overlap["osm_candidate_count"] = (osm_candidate_counts)

bike_loops_overlap["matched_osm_id"] = (matched_osm_ids)
bike_loops_overlap["lit"] = (matched_lit_values)
bike_loops_overlap["surface"] = (matched_surface_values)

bike_loops_overlap["lit_group"] = (
    bike_loops_overlap["lit"].apply(
        group_lit
    )
)

bike_loops_overlap["surface_group"] = (
    bike_loops_overlap["surface"].apply(
        group_surface
    )
)

result_columns = [
    column
    for column in [
        "edge_id",
        "node_start",
        "node_end",
        "bike_loop_length_m",
        "matched_length_m",
        "unmatched_length_m",
        "coverage_pct",
        "osm_candidate_count",
        "matched_osm_id",
        "lit",
        "lit_group",
        "surface",
        "surface_group",
    ]
    if column in bike_loops_overlap.columns
]

matched_edges = gpd.GeoDataFrame(
    bike_loops_overlap[result_columns].copy(),
    geometry=matched_geometries,
    crs=BIKE_LOOPS_CRS,
)

unmatched_edges = gpd.GeoDataFrame(
    bike_loops_overlap[result_columns].copy(),
    geometry=unmatched_geometries,
    crs=BIKE_LOOPS_CRS,
)

matched_edges = matched_edges.loc[~matched_edges.geometry.is_empty].copy()
unmatched_edges = unmatched_edges.loc[~unmatched_edges.geometry.is_empty].copy()

print(f"Matched edges: {len(matched_edges):,}")
print(f"Edges with unmatched geometry: {len(unmatched_edges):,}")

In [ ]:
overlap_summary = plot_overlap_results(
    sample=SAMPLE,
    tolerance_m=TOLERANCE_M,
    output_dir=OUTPUT_DIR,
    bike_loops_edges=bike_loops_edges,
    bike_loops_nodes=bike_loops_nodes,
    osm_edges=osm_edges,
    bike_loops_overlap=bike_loops_overlap,
    matched_edges=matched_edges,
    unmatched_edges=unmatched_edges,
    plot_padding_m=1_000,
)

In [ ]:
fig, axes = plot_matched_attribute(
    sample=SAMPLE,
    osm_edges=osm_edges,
    matched_edges=matched_edges,
    unmatched_edges=unmatched_edges,
    bike_loops_edges=bike_loops_edges,
    bike_loops_nodes=bike_loops_nodes,
    tolerance_m=TOLERANCE_M,
    output_dir=OUTPUT_DIR,
)

## Lollandfalster

In [ ]:
SAMPLE = "lollandfalster"

BIKE_LOOPS_DIR = PROJECT_ROOT / f"bikenwloops/data/input/{SAMPLE}/network/processed"
OUTPUT_DIR = PROJECT_ROOT / f"osm_data_exploration/outputs/{SAMPLE}"

OSM_EDGES_PATH = PROJECT_ROOT / f"osm_data_exploration/outputs/{SAMPLE}/{SAMPLE}_osm_bike_edges.parquet"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OSM_EDGES_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Bike loop network directory:", BIKE_LOOPS_DIR)
print("Output folder:", OUTPUT_DIR)

In [ ]:
# Load bike loop network data
bike_loops_edges = gpd.read_file(BIKE_LOOPS_DIR / "edges_slope.gpkg")
bike_loops_nodes = gpd.read_file(BIKE_LOOPS_DIR / "nodes.gpkg")

bike_loops_edges = bike_loops_edges.loc[
    bike_loops_edges.geometry.notna()
    & ~bike_loops_edges.geometry.is_empty
].copy()

bike_loops_nodes = bike_loops_nodes.loc[
    bike_loops_nodes.geometry.notna()
    & ~bike_loops_nodes.geometry.is_empty
].copy()

if bike_loops_edges.crs is None:
    raise ValueError("The bike-loop edges do not have a CRS.")

BIKE_LOOPS_CRS = bike_loops_edges.crs
bike_loops_nodes = bike_loops_nodes.to_crs(BIKE_LOOPS_CRS)

print("Bike loops CRS:", BIKE_LOOPS_CRS)
print(f"Bike loop edges: {len(bike_loops_edges):,}")
print(f"Bike loop nodes: {len(bike_loops_nodes):,}")

In [ ]:
minx, miny, maxx, maxy = bike_loops_edges.total_bounds

bike_loops_bbox = box(
    minx - EXTRACTION_MARGIN_M,
    miny - EXTRACTION_MARGIN_M,
    maxx + EXTRACTION_MARGIN_M,
    maxy + EXTRACTION_MARGIN_M,
)

# The bounding box below uses the BikeNodePlanner coordinate system (Bike loops CRS: EPSG:25832) with coordinates measured in metres. 
# This is useful for applying the 5,000-metre margin and performing the 20-metre overlap analysis.
bike_loops_bbox_gdf = gpd.GeoDataFrame(
    geometry=[bike_loops_bbox],
    crs=BIKE_LOOPS_CRS,
)

# WGS84 means the bounding box coordinates have been converted to the standard global longitude/latitude coordinate system, EPSG:4326.
bike_loops_bbox_wgs84 = (
    bike_loops_bbox_gdf
    .to_crs("EPSG:4326")
    .geometry
    .iloc[0]
)

### Extract area of interest from OSM data

In [ ]:
### Time: ≈ 2 min
required_osm_columns = {
    "geometry",
    "lit",
    "surface",
}

rebuild_osm_cache = True

if OSM_EDGES_PATH.exists():
    osm_edges = gpd.read_parquet(OSM_EDGES_PATH)
    missing_columns = (required_osm_columns - set(osm_edges.columns))
    
    if not missing_columns:
        rebuild_osm_cache = False
        print(f"Loaded cached {SAMPLE} OSM bicycle network.")

    else:
        print(f"The cached file is missing: {sorted(missing_columns)}")
        print("Rebuilding the OSM cache...")

if rebuild_osm_cache:
    print(f"Extracting the {SAMPLE} bicycle network from the Denmark PBF...")

    osm_reader = OSM(str(PBF_PATH), bounding_box=bike_loops_bbox_wgs84)

    osm_edges = osm_reader.get_network(
        network_type="cycling",
        extra_attributes=["lit", "surface"],
    )

    osm_edges = osm_edges.loc[
        osm_edges.geometry.notna()
        & ~osm_edges.geometry.is_empty
    ].copy()

    osm_edges = osm_edges.explode(
        index_parts=False,
        ignore_index=True,
    )

    osm_edges = osm_edges.to_crs(BIKE_LOOPS_CRS)

    columns_to_keep = [
        column
        for column in [
            "id",
            "name",
            "highway",
            "lit",
            "surface",
            "geometry",
        ]
        if column in osm_edges.columns
    ]

    osm_edges = osm_edges[columns_to_keep].copy()
    osm_edges.to_parquet(OSM_EDGES_PATH)
    print(f"Saved {SAMPLE.title()} OSM bicycle network.")

print(f"OSM bicycle edges: {len(osm_edges):,}")
print("OSM CRS:", osm_edges.crs)

In [ ]:
plot_bicycle_networks(
    sample=SAMPLE,
    bike_loops_edges=bike_loops_edges,
    bike_loops_nodes=bike_loops_nodes,
    bike_loops_bbox_gdf=bike_loops_bbox_gdf,
    osm_edges=osm_edges,
)

### Overlap analysis of the two bike networks

In [ ]:
osm_spatial_index = osm_edges.sindex
matched_geometries, unmatched_geometries, matched_lengths, osm_candidate_counts = [], [], [], []

matched_osm_ids, matched_lit_values, matched_surface_values = [], [], []
number_of_edges = len(bike_loops_edges)

for position, geometry in enumerate(bike_loops_edges.geometry, start=1):

    # Find only OSM lines within the matching tolerance
    candidate_positions = osm_spatial_index.query(
        geometry,
        predicate="dwithin", # This predicate finds geometries that are within a specified distance of the input geometry.
        distance=TOLERANCE_M,
    )

    candidate_positions = np.unique(candidate_positions)
    
    candidates = osm_edges.iloc[candidate_positions]

    if candidates.empty:
        matched_geometry = shapely.GeometryCollection()
        unmatched_geometry = geometry

        matched_osm_id = np.nan
        lit_value = np.nan
        surface_value = np.nan

    else:
        # Calculate the complete matched geometry
        local_osm_network = shapely.union_all(candidates.geometry.array)
        local_osm_buffer = local_osm_network.buffer(TOLERANCE_M)
        matched_geometry = geometry.intersection(local_osm_buffer)
        unmatched_geometry = geometry.difference(local_osm_buffer)
        
        # Find which candidate covers the greatest
        # length of this bike-loop edge
        candidate_buffers = (
            candidates.geometry.buffer(
                TOLERANCE_M
            )
        )

        candidate_overlap_lengths = (
            candidate_buffers
            .intersection(geometry)
            .length
        )

        best_position = (
            candidate_overlap_lengths
            .to_numpy()
            .argmax()
        )

        best_candidate = candidates.iloc[best_position]

        matched_osm_id = best_candidate.get("id", np.nan)
        lit_value = best_candidate.get("lit", np.nan)
        surface_value = best_candidate.get("surface", np.nan)

    matched_geometries.append(matched_geometry)
    unmatched_geometries.append(unmatched_geometry)
    matched_lengths.append(matched_geometry.length)
    osm_candidate_counts.append(len(candidates))
    matched_osm_ids.append(matched_osm_id)
    matched_lit_values.append(lit_value)
    matched_surface_values.append(surface_value)

    if position % 500 == 0:
        print(f"Processed {position:,} of {number_of_edges:,} bike-loop edges")

print("Matching completed.")

In [ ]:
bike_loops_overlap = bike_loops_edges.copy()
bike_loops_overlap["bike_loop_length_m"] = (bike_loops_overlap.geometry.length)
bike_loops_overlap["matched_length_m"] = (matched_lengths)
bike_loops_overlap["unmatched_length_m"] = (bike_loops_overlap["bike_loop_length_m"] - bike_loops_overlap["matched_length_m"])
bike_loops_overlap["coverage_pct"] = (100 * bike_loops_overlap["matched_length_m"] / bike_loops_overlap["bike_loop_length_m"])
bike_loops_overlap["osm_candidate_count"] = (osm_candidate_counts)

bike_loops_overlap["matched_osm_id"] = (matched_osm_ids)
bike_loops_overlap["lit"] = (matched_lit_values)
bike_loops_overlap["surface"] = (matched_surface_values)

bike_loops_overlap["lit_group"] = (
    bike_loops_overlap["lit"].apply(
        group_lit
    )
)

bike_loops_overlap["surface_group"] = (
    bike_loops_overlap["surface"].apply(
        group_surface
    )
)

result_columns = [
    column
    for column in [
        "edge_id",
        "node_start",
        "node_end",
        "bike_loop_length_m",
        "matched_length_m",
        "unmatched_length_m",
        "coverage_pct",
        "osm_candidate_count",
        "matched_osm_id",
        "lit",
        "lit_group",
        "surface",
        "surface_group",
    ]
    if column in bike_loops_overlap.columns
]

matched_edges = gpd.GeoDataFrame(
    bike_loops_overlap[result_columns].copy(),
    geometry=matched_geometries,
    crs=BIKE_LOOPS_CRS,
)

unmatched_edges = gpd.GeoDataFrame(
    bike_loops_overlap[result_columns].copy(),
    geometry=unmatched_geometries,
    crs=BIKE_LOOPS_CRS,
)

matched_edges = matched_edges.loc[~matched_edges.geometry.is_empty].copy()
unmatched_edges = unmatched_edges.loc[~unmatched_edges.geometry.is_empty].copy()

print(f"Matched edges: {len(matched_edges):,}")
print(f"Edges with unmatched geometry: {len(unmatched_edges):,}")

In [ ]:
overlap_summary = plot_overlap_results(
    sample=SAMPLE,
    tolerance_m=TOLERANCE_M,
    output_dir=OUTPUT_DIR,
    bike_loops_edges=bike_loops_edges,
    bike_loops_nodes=bike_loops_nodes,
    osm_edges=osm_edges,
    bike_loops_overlap=bike_loops_overlap,
    matched_edges=matched_edges,
    unmatched_edges=unmatched_edges,
    plot_padding_m=1_000,
)

In [ ]:
fig, axes = plot_matched_attribute(
    sample=SAMPLE,
    osm_edges=osm_edges,
    matched_edges=matched_edges,
    unmatched_edges=unmatched_edges,
    bike_loops_edges=bike_loops_edges,
    bike_loops_nodes=bike_loops_nodes,
    tolerance_m=TOLERANCE_M,
    output_dir=OUTPUT_DIR,
)

## Langeland

In [ ]:
SAMPLE = "longland"

BIKE_LOOPS_DIR = PROJECT_ROOT / f"bikenwloops/data/input/{SAMPLE}/network/processed"
OUTPUT_DIR = PROJECT_ROOT / f"osm_data_exploration/outputs/{SAMPLE}"

OSM_EDGES_PATH = PROJECT_ROOT / f"osm_data_exploration/outputs/{SAMPLE}/{SAMPLE}_osm_bike_edges.parquet"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OSM_EDGES_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Bike loop network directory:", BIKE_LOOPS_DIR)
print("Output folder:", OUTPUT_DIR)

In [ ]:
# Load bike loop network data
bike_loops_edges = gpd.read_file(BIKE_LOOPS_DIR / "edges_slope.gpkg")
bike_loops_nodes = gpd.read_file(BIKE_LOOPS_DIR / "nodes.gpkg")

bike_loops_edges = bike_loops_edges.loc[
    bike_loops_edges.geometry.notna()
    & ~bike_loops_edges.geometry.is_empty
].copy()

bike_loops_nodes = bike_loops_nodes.loc[
    bike_loops_nodes.geometry.notna()
    & ~bike_loops_nodes.geometry.is_empty
].copy()

if bike_loops_edges.crs is None:
    raise ValueError("The bike-loop edges do not have a CRS.")

BIKE_LOOPS_CRS = bike_loops_edges.crs
bike_loops_nodes = bike_loops_nodes.to_crs(BIKE_LOOPS_CRS)

print("Bike loops CRS:", BIKE_LOOPS_CRS)
print(f"Bike loop edges: {len(bike_loops_edges):,}")
print(f"Bike loop nodes: {len(bike_loops_nodes):,}")

In [ ]:
minx, miny, maxx, maxy = bike_loops_edges.total_bounds

bike_loops_bbox = box(
    minx - EXTRACTION_MARGIN_M,
    miny - EXTRACTION_MARGIN_M,
    maxx + EXTRACTION_MARGIN_M,
    maxy + EXTRACTION_MARGIN_M,
)

# The bounding box below uses the BikeNodePlanner coordinate system (Bike loops CRS: EPSG:25832) with coordinates measured in metres. 
# This is useful for applying the 5,000-metre margin and performing the 20-metre overlap analysis.
bike_loops_bbox_gdf = gpd.GeoDataFrame(
    geometry=[bike_loops_bbox],
    crs=BIKE_LOOPS_CRS,
)

# WGS84 means the bounding box coordinates have been converted to the standard global longitude/latitude coordinate system, EPSG:4326.
bike_loops_bbox_wgs84 = (
    bike_loops_bbox_gdf
    .to_crs("EPSG:4326")
    .geometry
    .iloc[0]
)

### Extract area of interest from OSM data

In [ ]:
### Time: ≈ 2-3 min
required_osm_columns = {
    "geometry",
    "lit",
    "surface",
}

rebuild_osm_cache = True

if OSM_EDGES_PATH.exists():
    osm_edges = gpd.read_parquet(OSM_EDGES_PATH)
    missing_columns = (required_osm_columns - set(osm_edges.columns))
    
    if not missing_columns:
        rebuild_osm_cache = False
        print(f"Loaded cached {SAMPLE} OSM bicycle network.")

    else:
        print(f"The cached file is missing: {sorted(missing_columns)}")
        print("Rebuilding the OSM cache...")

if rebuild_osm_cache:
    print(f"Extracting the {SAMPLE} bicycle network from the Denmark PBF...")

    osm_reader = OSM(str(PBF_PATH), bounding_box=bike_loops_bbox_wgs84)

    osm_edges = osm_reader.get_network(
        network_type="cycling",
        extra_attributes=["lit", "surface"],
    )

    osm_edges = osm_edges.loc[
        osm_edges.geometry.notna()
        & ~osm_edges.geometry.is_empty
    ].copy()

    osm_edges = osm_edges.explode(
        index_parts=False,
        ignore_index=True,
    )

    osm_edges = osm_edges.to_crs(BIKE_LOOPS_CRS)

    columns_to_keep = [
        column
        for column in [
            "id",
            "name",
            "highway",
            "lit",
            "surface",
            "geometry",
        ]
        if column in osm_edges.columns
    ]

    osm_edges = osm_edges[columns_to_keep].copy()
    osm_edges.to_parquet(OSM_EDGES_PATH)
    print(f"Saved {SAMPLE.title()} OSM bicycle network.")

print(f"OSM bicycle edges: {len(osm_edges):,}")
print("OSM CRS:", osm_edges.crs)

In [ ]:
plot_bicycle_networks(
    sample=SAMPLE,
    bike_loops_edges=bike_loops_edges,
    bike_loops_nodes=bike_loops_nodes,
    bike_loops_bbox_gdf=bike_loops_bbox_gdf,
    osm_edges=osm_edges,
)

### Overlap analysis of the two bike networks

In [ ]:
osm_spatial_index = osm_edges.sindex
matched_geometries, unmatched_geometries, matched_lengths, osm_candidate_counts = [], [], [], []

matched_osm_ids, matched_lit_values, matched_surface_values = [], [], []
number_of_edges = len(bike_loops_edges)

for position, geometry in enumerate(bike_loops_edges.geometry, start=1):

    # Find only OSM lines within the matching tolerance
    candidate_positions = osm_spatial_index.query(
        geometry,
        predicate="dwithin", # This predicate finds geometries that are within a specified distance of the input geometry.
        distance=TOLERANCE_M,
    )

    candidate_positions = np.unique(candidate_positions)
    
    candidates = osm_edges.iloc[candidate_positions]

    if candidates.empty:
        matched_geometry = shapely.GeometryCollection()
        unmatched_geometry = geometry

        matched_osm_id = np.nan
        lit_value = np.nan
        surface_value = np.nan

    else:
        # Calculate the complete matched geometry
        local_osm_network = shapely.union_all(candidates.geometry.array)
        local_osm_buffer = local_osm_network.buffer(TOLERANCE_M)
        matched_geometry = geometry.intersection(local_osm_buffer)
        unmatched_geometry = geometry.difference(local_osm_buffer)
        
        # Find which candidate covers the greatest
        # length of this bike-loop edge
        candidate_buffers = (
            candidates.geometry.buffer(
                TOLERANCE_M
            )
        )

        candidate_overlap_lengths = (
            candidate_buffers
            .intersection(geometry)
            .length
        )

        best_position = (
            candidate_overlap_lengths
            .to_numpy()
            .argmax()
        )

        best_candidate = candidates.iloc[best_position]

        matched_osm_id = best_candidate.get("id", np.nan)
        lit_value = best_candidate.get("lit", np.nan)
        surface_value = best_candidate.get("surface", np.nan)

    matched_geometries.append(matched_geometry)
    unmatched_geometries.append(unmatched_geometry)
    matched_lengths.append(matched_geometry.length)
    osm_candidate_counts.append(len(candidates))
    matched_osm_ids.append(matched_osm_id)
    matched_lit_values.append(lit_value)
    matched_surface_values.append(surface_value)

    if position % 500 == 0:
        print(f"Processed {position:,} of {number_of_edges:,} bike-loop edges")

print("Matching completed.")

In [ ]:
bike_loops_overlap = bike_loops_edges.copy()
bike_loops_overlap["bike_loop_length_m"] = (bike_loops_overlap.geometry.length)
bike_loops_overlap["matched_length_m"] = (matched_lengths)
bike_loops_overlap["unmatched_length_m"] = (bike_loops_overlap["bike_loop_length_m"] - bike_loops_overlap["matched_length_m"])
bike_loops_overlap["coverage_pct"] = (100 * bike_loops_overlap["matched_length_m"] / bike_loops_overlap["bike_loop_length_m"])
bike_loops_overlap["osm_candidate_count"] = (osm_candidate_counts)

bike_loops_overlap["matched_osm_id"] = (matched_osm_ids)
bike_loops_overlap["lit"] = (matched_lit_values)
bike_loops_overlap["surface"] = (matched_surface_values)

bike_loops_overlap["lit_group"] = (
    bike_loops_overlap["lit"].apply(
        group_lit
    )
)

bike_loops_overlap["surface_group"] = (
    bike_loops_overlap["surface"].apply(
        group_surface
    )
)

result_columns = [
    column
    for column in [
        "edge_id",
        "node_start",
        "node_end",
        "bike_loop_length_m",
        "matched_length_m",
        "unmatched_length_m",
        "coverage_pct",
        "osm_candidate_count",
        "matched_osm_id",
        "lit",
        "lit_group",
        "surface",
        "surface_group",
    ]
    if column in bike_loops_overlap.columns
]

matched_edges = gpd.GeoDataFrame(
    bike_loops_overlap[result_columns].copy(),
    geometry=matched_geometries,
    crs=BIKE_LOOPS_CRS,
)

unmatched_edges = gpd.GeoDataFrame(
    bike_loops_overlap[result_columns].copy(),
    geometry=unmatched_geometries,
    crs=BIKE_LOOPS_CRS,
)

matched_edges = matched_edges.loc[~matched_edges.geometry.is_empty].copy()
unmatched_edges = unmatched_edges.loc[~unmatched_edges.geometry.is_empty].copy()

print(f"Matched edges: {len(matched_edges):,}")
print(f"Edges with unmatched geometry: {len(unmatched_edges):,}")

In [ ]:
overlap_summary = plot_overlap_results(
    sample=SAMPLE,
    tolerance_m=TOLERANCE_M,
    output_dir=OUTPUT_DIR,
    bike_loops_edges=bike_loops_edges,
    bike_loops_nodes=bike_loops_nodes,
    osm_edges=osm_edges,
    bike_loops_overlap=bike_loops_overlap,
    matched_edges=matched_edges,
    unmatched_edges=unmatched_edges,
    plot_padding_m=1_000,
)

In [ ]:
fig, axes = plot_matched_attribute(
    sample=SAMPLE,
    osm_edges=osm_edges,
    matched_edges=matched_edges,
    unmatched_edges=unmatched_edges,
    bike_loops_edges=bike_loops_edges,
    bike_loops_nodes=bike_loops_nodes,
    tolerance_m=TOLERANCE_M,
    output_dir=OUTPUT_DIR,
)

## Sjælland

In [ ]:
SAMPLE = "zealand"

BIKE_LOOPS_DIR = PROJECT_ROOT / f"bikenwloops/data/input/{SAMPLE}/network/processed"
OUTPUT_DIR = PROJECT_ROOT / f"osm_data_exploration/outputs/{SAMPLE}"

OSM_EDGES_PATH = PROJECT_ROOT / f"osm_data_exploration/outputs/{SAMPLE}/{SAMPLE}_osm_bike_edges.parquet"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OSM_EDGES_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Bike loop network directory:", BIKE_LOOPS_DIR)
print("Output folder:", OUTPUT_DIR)

In [ ]:
# Load bike loop network data
bike_loops_edges = gpd.read_file(BIKE_LOOPS_DIR / "edges_slope.gpkg")
bike_loops_nodes = gpd.read_file(BIKE_LOOPS_DIR / "nodes.gpkg")

bike_loops_edges = bike_loops_edges.loc[
    bike_loops_edges.geometry.notna()
    & ~bike_loops_edges.geometry.is_empty
].copy()

bike_loops_nodes = bike_loops_nodes.loc[
    bike_loops_nodes.geometry.notna()
    & ~bike_loops_nodes.geometry.is_empty
].copy()

if bike_loops_edges.crs is None:
    raise ValueError("The bike-loop edges do not have a CRS.")

BIKE_LOOPS_CRS = bike_loops_edges.crs
bike_loops_nodes = bike_loops_nodes.to_crs(BIKE_LOOPS_CRS)

print("Bike loops CRS:", BIKE_LOOPS_CRS)
print(f"Bike loop edges: {len(bike_loops_edges):,}")
print(f"Bike loop nodes: {len(bike_loops_nodes):,}")

In [ ]:
minx, miny, maxx, maxy = bike_loops_edges.total_bounds

bike_loops_bbox = box(
    minx - EXTRACTION_MARGIN_M,
    miny - EXTRACTION_MARGIN_M,
    maxx + EXTRACTION_MARGIN_M,
    maxy + EXTRACTION_MARGIN_M,
)

# The bounding box below uses the BikeNodePlanner coordinate system (Bike loops CRS: EPSG:25832) with coordinates measured in metres. 
# This is useful for applying the 5,000-metre margin and performing the 20-metre overlap analysis.
bike_loops_bbox_gdf = gpd.GeoDataFrame(
    geometry=[bike_loops_bbox],
    crs=BIKE_LOOPS_CRS,
)

# WGS84 means the bounding box coordinates have been converted to the standard global longitude/latitude coordinate system, EPSG:4326.
bike_loops_bbox_wgs84 = (
    bike_loops_bbox_gdf
    .to_crs("EPSG:4326")
    .geometry
    .iloc[0]
)

### Extract area of interest from OSM data

In [ ]:
### Time: ≈ 17 min
required_osm_columns = {
    "geometry",
    "lit",
    "surface",
}

rebuild_osm_cache = True

if OSM_EDGES_PATH.exists():
    osm_edges = gpd.read_parquet(OSM_EDGES_PATH)
    missing_columns = (required_osm_columns - set(osm_edges.columns))
    
    if not missing_columns:
        rebuild_osm_cache = False
        print(f"Loaded cached {SAMPLE} OSM bicycle network.")

    else:
        print(f"The cached file is missing: {sorted(missing_columns)}")
        print("Rebuilding the OSM cache...")

if rebuild_osm_cache:
    print(f"Extracting the {SAMPLE} bicycle network from the Denmark PBF...")

    osm_reader = OSM(str(PBF_PATH), bounding_box=bike_loops_bbox_wgs84)

    osm_edges = osm_reader.get_network(
        network_type="cycling",
        extra_attributes=["lit", "surface"],
    )

    osm_edges = osm_edges.loc[
        osm_edges.geometry.notna()
        & ~osm_edges.geometry.is_empty
    ].copy()

    osm_edges = osm_edges.explode(
        index_parts=False,
        ignore_index=True,
    )

    osm_edges = osm_edges.to_crs(BIKE_LOOPS_CRS)

    columns_to_keep = [
        column
        for column in [
            "id",
            "name",
            "highway",
            "lit",
            "surface",
            "geometry",
        ]
        if column in osm_edges.columns
    ]

    osm_edges = osm_edges[columns_to_keep].copy()
    osm_edges.to_parquet(OSM_EDGES_PATH)
    print(f"Saved {SAMPLE.title()} OSM bicycle network.")

print(f"OSM bicycle edges: {len(osm_edges):,}")
print("OSM CRS:", osm_edges.crs)

In [ ]:
plot_bicycle_networks(
    sample=SAMPLE,
    bike_loops_edges=bike_loops_edges,
    bike_loops_nodes=bike_loops_nodes,
    bike_loops_bbox_gdf=bike_loops_bbox_gdf,
    osm_edges=osm_edges,
)

### Overlap analysis of the two bike networks

In [ ]:
osm_spatial_index = osm_edges.sindex
matched_geometries, unmatched_geometries, matched_lengths, osm_candidate_counts = [], [], [], []

matched_osm_ids, matched_lit_values, matched_surface_values = [], [], []
number_of_edges = len(bike_loops_edges)

for position, geometry in enumerate(bike_loops_edges.geometry, start=1):

    # Find only OSM lines within the matching tolerance
    candidate_positions = osm_spatial_index.query(
        geometry,
        predicate="dwithin", # This predicate finds geometries that are within a specified distance of the input geometry.
        distance=TOLERANCE_M,
    )

    candidate_positions = np.unique(candidate_positions)
    
    candidates = osm_edges.iloc[candidate_positions]

    if candidates.empty:
        matched_geometry = shapely.GeometryCollection()
        unmatched_geometry = geometry

        matched_osm_id = np.nan
        lit_value = np.nan
        surface_value = np.nan

    else:
        # Calculate the complete matched geometry
        local_osm_network = shapely.union_all(candidates.geometry.array)
        local_osm_buffer = local_osm_network.buffer(TOLERANCE_M)
        matched_geometry = geometry.intersection(local_osm_buffer)
        unmatched_geometry = geometry.difference(local_osm_buffer)
        
        # Find which candidate covers the greatest
        # length of this bike-loop edge
        candidate_buffers = (
            candidates.geometry.buffer(
                TOLERANCE_M
            )
        )

        candidate_overlap_lengths = (
            candidate_buffers
            .intersection(geometry)
            .length
        )

        best_position = (
            candidate_overlap_lengths
            .to_numpy()
            .argmax()
        )

        best_candidate = candidates.iloc[best_position]

        matched_osm_id = best_candidate.get("id", np.nan)
        lit_value = best_candidate.get("lit", np.nan)
        surface_value = best_candidate.get("surface", np.nan)

    matched_geometries.append(matched_geometry)
    unmatched_geometries.append(unmatched_geometry)
    matched_lengths.append(matched_geometry.length)
    osm_candidate_counts.append(len(candidates))
    matched_osm_ids.append(matched_osm_id)
    matched_lit_values.append(lit_value)
    matched_surface_values.append(surface_value)

    if position % 500 == 0:
        print(f"Processed {position:,} of {number_of_edges:,} bike-loop edges")

print("Matching completed.")

In [ ]:
bike_loops_overlap = bike_loops_edges.copy()
bike_loops_overlap["bike_loop_length_m"] = (bike_loops_overlap.geometry.length)
bike_loops_overlap["matched_length_m"] = (matched_lengths)
bike_loops_overlap["unmatched_length_m"] = (bike_loops_overlap["bike_loop_length_m"] - bike_loops_overlap["matched_length_m"])
bike_loops_overlap["coverage_pct"] = (100 * bike_loops_overlap["matched_length_m"] / bike_loops_overlap["bike_loop_length_m"])
bike_loops_overlap["osm_candidate_count"] = (osm_candidate_counts)

bike_loops_overlap["matched_osm_id"] = (matched_osm_ids)
bike_loops_overlap["lit"] = (matched_lit_values)
bike_loops_overlap["surface"] = (matched_surface_values)

bike_loops_overlap["lit_group"] = (
    bike_loops_overlap["lit"].apply(
        group_lit
    )
)

bike_loops_overlap["surface_group"] = (
    bike_loops_overlap["surface"].apply(
        group_surface
    )
)

result_columns = [
    column
    for column in [
        "edge_id",
        "node_start",
        "node_end",
        "bike_loop_length_m",
        "matched_length_m",
        "unmatched_length_m",
        "coverage_pct",
        "osm_candidate_count",
        "matched_osm_id",
        "lit",
        "lit_group",
        "surface",
        "surface_group",
    ]
    if column in bike_loops_overlap.columns
]

matched_edges = gpd.GeoDataFrame(
    bike_loops_overlap[result_columns].copy(),
    geometry=matched_geometries,
    crs=BIKE_LOOPS_CRS,
)

unmatched_edges = gpd.GeoDataFrame(
    bike_loops_overlap[result_columns].copy(),
    geometry=unmatched_geometries,
    crs=BIKE_LOOPS_CRS,
)

matched_edges = matched_edges.loc[~matched_edges.geometry.is_empty].copy()
unmatched_edges = unmatched_edges.loc[~unmatched_edges.geometry.is_empty].copy()

print(f"Matched edges: {len(matched_edges):,}")
print(f"Edges with unmatched geometry: {len(unmatched_edges):,}")

In [ ]:
overlap_summary = plot_overlap_results(
    sample=SAMPLE,
    tolerance_m=TOLERANCE_M,
    output_dir=OUTPUT_DIR,
    bike_loops_edges=bike_loops_edges,
    bike_loops_nodes=bike_loops_nodes,
    osm_edges=osm_edges,
    bike_loops_overlap=bike_loops_overlap,
    matched_edges=matched_edges,
    unmatched_edges=unmatched_edges,
    plot_padding_m=1_000,
)

In [ ]:
fig, axes = plot_matched_attribute(
    sample=SAMPLE,
    osm_edges=osm_edges,
    matched_edges=matched_edges,
    unmatched_edges=unmatched_edges,
    bike_loops_edges=bike_loops_edges,
    bike_loops_nodes=bike_loops_nodes,
    tolerance_m=TOLERANCE_M,
    output_dir=OUTPUT_DIR,
)